# Lesson 6, Part 2

Here is code from Level 5 Part 2 that develops a Taylor expansion of exp( ) using Polynomial, 
along with a faster version that sums terms directly.

In [ ]:
# initializations
import numpy as np
import math
import matplotlib.pyplot as plt
from numpy.polynomial import Polynomial as P
from time import time as T

def taylor_exp(order,x):
    inv_factorial = 1
    coef = np.ones(order+1)
    for n in range(1,order+1):
        inv_factorial /=  n
        coef[n] = inv_factorial
    return P(coef)(x)

def taylor_exp_fast(order,x):
    estimate = new_term = 1
    for n in range(1,order+1):
        new_term *= x/n
        estimate += new_term
    return estimate  

for order,x in ((20,5),(30,10)):
    print('Comparisons for exp(',x,'):')
    iters,tic = 1000,T()
    for iter in range(iters): taylor_exp(order,x)
    print(taylor_exp(order,x),'using taylor_exp with order',order,', in time',T()-tic); tic=T()
    for iter in range(iters): taylor_exp_fast(order,x)
    print(taylor_exp_fast(order,x),'using taylor_exp_fast with order',order,', in time',T()-tic,'\n')

The main problem with the previous code is that it is hard to gauge how high the order needs to be. 
The code here determines order endogenously, although it runs slower as a result.

In [ ]:
def taylor_exp_flexible(x,tol=1E-16):
    estimate = new_term = 1
    for n in range(1,150):
        new_term *= x/n
        estimate += new_term
        if abs(new_term)  < tol * estimate:
            break
    return estimate,n 

for x in (3,30):  
    print('Comparisons for exp(',x,'):')
    order,iters,tic = 40,1000,T()
    for iter in range(iters): taylor_exp_fast(order,x)
    print(taylor_exp_fast(order,x),'using taylor_exp_fast with order',order,', in time',T()-tic); tic = T()
    order = 120
    for iter in range(iters): taylor_exp_fast(order,x)
    print(taylor_exp_fast(order,x),'using taylor_exp_fast with order',order,', in time',T()-tic); tic = T()
    estimate,n = taylor_exp_flexible(x)
    for iter in range(iters): taylor_exp_flexible(x)
    print(estimate,'using taylor_exp_flexible with order',n,', in time',T()-tic,'\n')

Recall that Python stores numbers in the form *mantissa* 2***exponent*, where 0.5 <= *mantissa* < 1
and *exponent* is an integer. The code here computes exp(*mantissa*), 
which reaches float precision by order 18. 
It then squares the result *exponent* times.

In [ ]:
inv_factorial = 1
tol = 2E-16
for n in range(1,100):
    inv_factorial /= n
    if inv_factorial < tol:
        break
print('For mantissa < 1, order',n,'suffices to make last term <',tol)

def taylor_exp_better(x,tol=1E-15):
    mantissa,exponent = math.frexp(x)
    new_term = mantissa
    estimate = 1 + new_term 
    for n in range(2,19):
        new_term *= mantissa/n
        estimate += new_term
    for m in range(exponent):
        estimate *= estimate
    return estimate,n  

x = 30
print('Comparisons for exp(',x,'):')
iters,tic=1000,T()
for iter in range(iters): taylor_exp_flexible(x)
print(taylor_exp_flexible(x),'using taylor_exp_flexible in time',T()-tic); tic = T()
for iter in range(iters): taylor_exp_better(x)
print(taylor_exp_better(x),'using taylor_exp_better in time',T()-tic)

The answers correspond closely to those using the commands **exp** in numpy or math. 
However, the built-in commands are much faster, and numpy can handle ndarrays.

In [ ]:
def taylor_exp_better(x,tol=1E-15):
    mantissa,exponent = math.frexp(x)
    new_term = mantissa
    estimate = 1 + new_term 
    for n in range(2,19):
        new_term *= mantissa/n
        estimate += new_term
    for m in range(exponent):
        estimate *= estimate
    return estimate

In [ ]:
num = 1000
max_rel_diverge = sum_rel_diverge = 0
for x in np.linspace(0.5,10,num):
    estimate = taylor_exp_better(x)
    rel_diverge = abs(estimate/math.exp(x) - 1)
    if rel_diverge > max_rel_diverge:
        max_rel_diverge = rel_diverge
    sum_rel_diverge += rel_diverge
rd_avg = round(sum_rel_diverge/num,16)
rd_max = round(max_rel_diverge,16)
print('Relative divergence of taylor_exp_better from math.exp averages',rd_avg,'with maximum',rd_max)
iters = 10000
for x in 3,30:
    print('\nComparisons for exp(',x,'):')
    tic = T()
    for iter in range(iters): np.exp(x)
    print(np.exp(x),'using np.exp in time',T()-tic); tic = T();
    for iter in range(iters): math.exp(x)
    print(math.exp(x),'using math.exp in time',T()-tic)

The value *e*=exp(1)  is so important that numpy and math store it as **_e_**.

In [ ]:
print('Relative divergence from true e:')
n = 1
for m in range(1,15):
    n *= 2
    e_approx = 1 + 1/(n-0.5)
    for k in range(m):
        e_approx *= e_approx
    rel_diverge = e_approx/math.e - 1
    print('n =',n,':',round(rel_diverge,10))

The text showed that *e* = lim(1 + 1/(*n*-0.5))<sup>*n*</sup> as *n* approaches infinity. 
Let's check this for *n* = 2<sup>*m*</sup>, where *m* is a whole number.

In [ ]:
print('Relative divergence from true e:')
n = 1
for m in range(1,15):
    n *= 2
    e_approx = 1 + 1/(n - 0.5)
    for k in range(m):
        e_approx *= e_approx
    rel_diverge = e_approx/math.e - 1
    print('n =',n,':',round(rel_diverge,10))

In the preceding computations, every doubling of *n* 
shaves the divergence by roughly three-quarters.
if we use the more standard approximation *e*=lim(1+1/*n*)<sup>*n*</sup>, 
each doubling of *n* not quite halves the divergence, so convergence takes twice as long.

In [ ]:
print('Relative divergence from true e:')
n = 1
for m in range(1,15):
    n *= 2
    e_approx = 1 + 1/(n-0.5)
    e_approx2 = 1 + 1/n
    for k in range(m):
        e_approx *= e_approx
        e_approx2 *= e_approx2
    rel_diverge = e_approx/np.e - 1
    rel_diverge2 = e_approx2/np.e - 1
    print('n =',n,':',round(rel_diverge,6),'   ',round(rel_diverge2,6))

The standard Taylor expansion for exp(1) converges much faster.

In [ ]:
print('Relative divergence from true e for Taylor expansion:')
max_order = 18
estimate = new_term = 1
for n in range(1,max_order+1):
    new_term *= 1/n
    estimate += new_term
    rel_diverge = estimate/np.e - 1
    print('order =',n,':',round(rel_diverge,17))

Compared to modified power functions (1 + *x*)<sup>*n*</sup> with *x* > 0 and *n* > 1, 
exp(*x*) rises more slowly at first but eventually surpasses them.

In [ ]:
power = 4  # experiment with different powers
num = 100
x = np.linspace(0,power**2,num)
y = np.exp(x)
z = 1
for i in range(power):
    z *= 1 + x
half = slice(0,num//2)
plt.style.use("seaborn-v0_8")
plt.subplot(1,2,1)
plt.title('Exp(x) versus (1+x)**'+str(power))
plt.plot(x[half],y[half],x[half],z[half])
plt.legend(labels=('exp(x)','(1+x)**'+str(power)))
plt.subplot(1,2,2)
plt.title('Exp(x) versus (1+x)**'+str(power))
plt.plot(x,y,x,z)
plt.legend(labels=('exp(x)','(1+x)**'+str(power)))
plt.show()

This is clearer when plotted in log terms. The logarithm of the power function 
is concave (slope decreasing) while the logarithm of exp(*x*) is straight.

In [ ]:
power = 4  # experiment with different powers
num = 100
x = np.linspace(0,power**2,num)
y = np.exp(x)
z = 1
for i in range(power):
    z *= 1 + x
half = slice(0,num//2)
plt.title('Exp(x) versus (1+x)**'+str(power)+' in log terms')
plt.semilogy(x,np.exp(x),label='exp(x)')
plt.semilogy(x,(1+x)**power,label='(1+x)**'+str(power))
plt.legend()
plt.show()

In fact, **exp**(**ln**(*x*)) = *x*, as **exp** and **ln** are inverse functions. 
Let's check that with computations.

In [ ]:
x = np.geomspace(1E-30,1E30,1000)  # keep the ratio of neighboring terms constant
diverge = np.exp(np.log(x)) - x
abs_rel_diverge = np.abs(diverge) / x
mean_rd = np.mean(abs_rel_diverge)   # np.mean computes the average value of an ndarray
max_rd = np.max(abs_rel_diverge)
print('Relative divergence of ln(exp(x)) from x averages',round(mean_rd,16),'with maximum',round(max_rd,16))

Growth at continuous fractional rate *g* will multiply the value by exp(*gt*) at time *t*. 
In comparison, growth at fractional rate *G* per period will multiply the value by (1+*G*)<sup>*t*</sup> at period *t*. 
Let's compare the times needed at various *g* or *G* for population or income to double.

In [ ]:
def doubling_time(growth_rate,isContinuous):
    if isContinuous:
        return np.log(2) / growth_rate
    else:
        return np.log(2) / np.log(1 + growth_rate)

print('Years needed to double for various growth rates g in years')
print('  g  Continuous  Yearly  Rule of 70')
for g in np.linspace(0.01,0.15,15):
    years1 = round(doubling_time(g,True),1)
    years2 = round(doubling_time(g,False),1)
    years3 = round(0.70 / g,1)
    print(round(g,2),'   ',years1,'    ',years2,'    ',years3)

As usual in our printouts, the columns get misaligned when leading or trailing zeros get dropped. 
Let's remedy that using a few of the many commands Python offers for formatting strings. 
Let's start with the command '{0:*n*}'.format(*y*), 
which says to format the variable *y* into a string of *n* characters.

In [ ]:
print('Years needed to double for various growth rates g in years')
print('  g  Continuous  Yearly  Rule of 70')
for g in np.linspace(0.01,0.15,15):
    years1 = round(doubling_time(g,True),1)
    years2 = round(doubling_time(g,False),1)
    years3 = round(0.70 / g,1)
    print('{0:4}'.format(round(g,2)),'   ','{0:4}'.format(years1),'    ','{0:4}'.format(years2),'    ','{0:4}'.format(years3))

The preceding code worked very well, 
except that the value of 0.1 for *g* is better written as 0.10. 
We can fix that by adding '.2f' inside the right bracket '}', 
which allots two digits to the right of the decimal point.
That does rounding as well, so if we add '.2f' or '.1f' we can drop all the **round** commands.

In [ ]:
print('Years needed to double for various growth rates g in years')
print('  g  Continuous  Yearly  Rule of 70')
for g in np.linspace(0.01,0.15,15):
    years1 = round(doubling_time(g,True),1)
    years2 = round(doubling_time(g,False),1)
    years3 = round(0.70 / g,1)
    # print('{0:4.2f}'.format(g),'   ','{0:4.1f}'.format(years1),'    ','{0:4.1f}'.format(years2),'    ','{0:4.1f}'.format(years3))
    # format '{0:n}' reserves n spaces even if not needed, so we can repalce preceding line with linel below
    print('{0:4.2f}'.format(g),'{0:8.1f}'.format(years1),'{0:9.1f}'.format(years2),'{0:9.1f}'.format(years3))